# Step 1 of RAG: Document Loading and Chunking Strategies

RAG (Retrieval-Augmented Generation) has 4 core steps:
1. **Document Loading and Chunking** (this notebook)
2. Generating embeddings
3. Storing in a vector database
4. Retrieval and passing context to an LLM

### What you will learn
1. Why chunking matters
2. Loading `.txt` and `.pdf` documents
3. Five chunking strategies (fixed-size, `CharacterTextSplitter`, `RecursiveCharacterTextSplitter`, sentence-based, token-based)
4. Comparing all strategies

## Why Chunking Matters
- Embedding models have a maximum input length (often 256 to 512 tokens).
- One giant embedding blurs many ideas together and gives poor search results.
- Smaller, focused chunks give more accurate embeddings and retrieval.

In [ ]:
!pip install -q langchain-text-splitters tiktoken pypdf fpdf2 pandas matplotlib

## 1. Create a Sample Document
We generate a file locally so the notebook runs without external downloads.

In [ ]:
import os
os.makedirs("docs", exist_ok=True)

article_text = """Space Exploration: A Brief History

Space exploration began in 1957 when the Soviet Union launched Sputnik 1, the first artificial satellite. This event started the Space Race between the United States and the Soviet Union and led to rapid advances in rocket technology.

In 1969, NASA's Apollo 11 mission landed the first humans on the Moon. Neil Armstrong and Buzz Aldrin walked on the lunar surface while Michael Collins orbited above. The mission proved that humans could travel to another world and return safely.

India entered the field through the Indian Space Research Organisation, or ISRO. In 2014, the Mars Orbiter Mission made India the first country to reach Mars orbit on its first attempt. In 2023, Chandrayaan-3 landed near the Moon's south pole, a region no other mission had reached.

Modern space exploration also involves private companies. Reusable rockets have greatly reduced launch costs, which makes it cheaper to place satellites in orbit. These satellites provide internet access, weather forecasts, and navigation services to billions of people.

Looking ahead, space agencies are planning crewed missions to Mars and permanent bases on the Moon. Scientists are also searching for signs of life on distant planets using powerful space telescopes.
"""

with open("docs/space_article.txt", "w") as f:
    f.write(article_text)

print("Characters:", len(article_text), "| Words:", len(article_text.split()))

## 2. Loading a `.txt` Document

In [ ]:
with open("docs/space_article.txt", "r") as f:
    loaded_text = f.read()

print(loaded_text[:300], "...")

## 3. Loading a `.pdf` Document

In [ ]:
from fpdf import FPDF

pdf = FPDF()
pdf.add_page()
pdf.set_font("Times", size=12)
for line in article_text.split("\n"):
    pdf.multi_cell(0, 8, line, new_x="LMARGIN", new_y="NEXT")
pdf.output("docs/space_article.pdf")
print("Created docs/space_article.pdf")

In [ ]:
from pypdf import PdfReader

reader = PdfReader("docs/space_article.pdf")
pdf_text = "\n".join(page.extract_text() for page in reader.pages)

print("Pages:", len(reader.pages))
print(pdf_text[:300], "...")

Loading depends on the file format, but the goal is always the same: **plain text**. From here on we chunk `loaded_text` (from the `.txt` file).

## 4. Strategy 1: Fixed-Size Character Chunking (from scratch)

In [ ]:
def fixed_size_chunking(text, chunk_size=300, overlap=50):
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start + chunk_size])
        start += chunk_size - overlap
    return chunks

fixed_chunks = fixed_size_chunking(loaded_text, chunk_size=300, overlap=50)
print("Number of chunks:", len(fixed_chunks))
for i in range(2):
    print(f"\n--- Chunk {i+1} ({len(fixed_chunks[i])} chars) ---\n{fixed_chunks[i]}")

Notice chunks can end in the middle of a word or sentence, since this method ignores text structure.

## 5. Strategy 2: `CharacterTextSplitter` (LangChain)

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

char_splitter = CharacterTextSplitter(separator="\n\n", chunk_size=300, chunk_overlap=50)
char_chunks = char_splitter.split_text(loaded_text)

print("Number of chunks:", len(char_chunks))
for i, c in enumerate(char_chunks):
    print(f"\n--- Chunk {i+1} ({len(c)} chars) ---\n{c}")

This splitter only splits on the chosen separator (blank line), so a paragraph longer than `chunk_size` stays as one big chunk (LangChain may print a warning about it).

## 6. Strategy 3: `RecursiveCharacterTextSplitter` (recommended default)
It tries paragraph breaks first, then lines, then sentences, then words, and only cuts characters as a last resort.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300, chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""],
)
recursive_chunks = recursive_splitter.split_text(loaded_text)

print("Number of chunks:", len(recursive_chunks))
for i, c in enumerate(recursive_chunks):
    print(f"\n--- Chunk {i+1} ({len(c)} chars) ---\n{c}")

## 7. Strategy 4: Sentence-Based Chunking

In [ ]:
import re

def sentence_chunking(text, sentences_per_chunk=2):
    sentences = [s for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s]
    return [" ".join(sentences[i:i + sentences_per_chunk])
            for i in range(0, len(sentences), sentences_per_chunk)]

sentence_chunks = sentence_chunking(loaded_text, sentences_per_chunk=2)
print("Number of chunks:", len(sentence_chunks))
for i, c in enumerate(sentence_chunks[:3]):
    print(f"\n--- Chunk {i+1} ---\n{c}")

## 8. Strategy 5: Token-Based Chunking
Models limit input in **tokens** (roughly 4 characters each in English), so token chunking guarantees the limit is respected.

In [ ]:
import tiktoken

encoding = tiktoken.get_encoding("cl100k_base")

def token_chunking(text, max_tokens=60, overlap_tokens=10):
    tokens = encoding.encode(text)
    chunks, start = [], 0
    while start < len(tokens):
        chunks.append(encoding.decode(tokens[start:start + max_tokens]))
        start += max_tokens - overlap_tokens
    return chunks

token_chunks = token_chunking(loaded_text, max_tokens=60, overlap_tokens=10)
print("Number of chunks:", len(token_chunks))
for i, c in enumerate(token_chunks[:3]):
    print(f"\n--- Chunk {i+1} ({len(encoding.encode(c))} tokens) ---\n{c}")

## 9. Compare All Strategies

In [ ]:
import pandas as pd

strategies = {
    "Fixed-size (scratch)": fixed_chunks,
    "CharacterTextSplitter": char_chunks,
    "RecursiveCharacterTextSplitter": recursive_chunks,
    "Sentence-based": sentence_chunks,
    "Token-based": token_chunks,
}

comparison = pd.DataFrame({
    "Strategy": list(strategies),
    "Number of Chunks": [len(c) for c in strategies.values()],
    "Avg Length (chars)": [round(sum(map(len, c)) / len(c), 1) for c in strategies.values()],
    "Max Length (chars)": [max(map(len, c)) for c in strategies.values()],
})
comparison

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.bar(comparison["Strategy"], comparison["Number of Chunks"])
plt.xticks(rotation=30, ha="right")
plt.ylabel("Number of chunks")
plt.title("Chunks produced by each strategy")
plt.tight_layout()
plt.show()

## 10. Practice Exercises (Solved)

In [ ]:
# Exercise 1: effect of chunk_size on RecursiveCharacterTextSplitter
for size in [600, 300, 150]:
    s = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=size // 6)
    print(f"chunk_size={size:4d} -> {len(s.split_text(loaded_text))} chunks")

In [ ]:
# Exercise 2: sentences_per_chunk = 1, 2, 4
for n in [1, 2, 4]:
    print(f"sentences_per_chunk={n} -> {len(sentence_chunking(loaded_text, n))} chunks")

In [ ]:
# Exercise 3: run all 5 strategies on the text extracted from the PDF
pdf_strategies = {
    "Fixed-size": fixed_size_chunking(pdf_text),
    "Character": char_splitter.split_text(pdf_text),
    "Recursive": recursive_splitter.split_text(pdf_text),
    "Sentence": sentence_chunking(pdf_text),
    "Token": token_chunking(pdf_text),
}
for name, chunks in pdf_strategies.items():
    print(f"{name:12s} -> {len(chunks)} chunks")

`CharacterTextSplitter` returns very few chunks here because PDF extraction usually separates paragraphs with single line breaks, not the blank line (`\n\n`) the splitter looks for. `RecursiveCharacterTextSplitter` still works because it falls back to other separators. This is a good reason to prefer it.

## Key Takeaways

| Strategy | Best for | Watch out for |
|---|---|---|
| Fixed-size | Learning the core idea | Cuts words and sentences |
| `CharacterTextSplitter` | Simple separator-based splitting | Rigid, big paragraphs stay big |
| `RecursiveCharacterTextSplitter` | Most RAG use cases | Slightly more setup |
| Sentence-based | Precision-sensitive text | Uneven chunk sizes |
| Token-based | Respecting model token limits | Needs a tokenizer |

`chunk_overlap` repeats a little text between neighbouring chunks so ideas on a boundary are not lost.
The next RAG step is turning each chunk into an embedding and storing it in a vector database.